Imports and Setup

In [1]:
import boto3
from tqdm.notebook import tqdm

# Initialize S3 and Rekognition clients
s3 = boto3.client('s3')
rekognition = boto3.client('rekognition')
bucket_name = 'thumbnail-lake'

In [2]:
%load_ext sql
%reload_ext sql
%config SqlMagic.displaylimit = None
%sql sqlite:///../data/thumbnail_model.db
%config SqlMagic.feedback = 0

displaylimit: Value None will be treated as 0 (no limit)

displaylimit: Value None will be treated as 0 (no limit)

Connecting to 'sqlite:///../data/thumbnail_model.db'

Upload Thumbnails to S3

In [4]:
import os

# paginate through all the objects in the bucket
response = s3.list_objects_v2(Bucket=bucket_name)
paginator = s3.get_paginator('list_objects_v2')
page_iterator = paginator.paginate(Bucket=bucket_name)
existing_keys = [obj['Key'] for page in page_iterator for obj in page.get('Contents', [])]

# Determine which thumbnails need to be uploaded
local_thumbnail_dir = f'../data/thumbnails/'
os.makedirs(local_thumbnail_dir, exist_ok=True)
local_files = [f for f in os.listdir(local_thumbnail_dir) if f.endswith('.jpg')]
upload_keys = [key for key in local_files if key not in existing_keys]

print(f'Uploading {len(upload_keys)} new thumbnails to S3...')

# Upload missing thumbnails to S3
for key in tqdm(upload_keys, desc="Uploading thumbnails"):
    s3.upload_file(f"{local_thumbnail_dir}/{key}", bucket_name, key)

Uploading 0 new thumbnails to S3...


Uploading thumbnails: 0it [00:00, ?it/s]

Define Rekognition Functions

Analyze Thumbnails with Rekognition and Save Responses

In [4]:
%%sql
CREATE TABLE IF NOT EXISTS rekognition_responses (
    video_id TEXT PRIMARY KEY,
    label_response TEXT,
    text_response TEXT
)

++
||
++
++

In [5]:
import asyncio
import aiosqlite
import boto3
import json
import nest_asyncio
from tqdm.notebook import tqdm

# Apply nest_asyncio
nest_asyncio.apply()


def get_image_labels(bucket_name, key, client):
    label_response = client.detect_labels(
        Image={'S3Object': {'Bucket': bucket_name, 'Name': key}},
        MaxLabels=32,  # When looking to improve performance, consider these two numbers
        MinConfidence=55,
        Features=['GENERAL_LABELS', 'IMAGE_PROPERTIES'],
        Settings={'ImageProperties': {'MaxDominantColors': 10}}
    )
    return label_response

def get_image_text(bucket_name, key, client):
    text_response = client.detect_text(
        Image={'S3Object': {'Bucket': bucket_name, 'Name': key}}
    )
    return text_response

# Synchronous function to get Rekognition response
def get_image_response_sync(bucket_name, key, client):
    label_response = get_image_labels(bucket_name, key, client)
    text_response = get_image_text(bucket_name, key, client)
    combined_response = {
        'Label Response': label_response,
        'Text Response': text_response
    }
    return combined_response

# Function to insert data into the database asynchronously
async def insert_into_db(conn, key, label_response, text_response):
    async with conn.execute(
        "INSERT INTO rekognition_responses (video_id, label_response, text_response) VALUES (?, ?, ?)",
        (key, label_response, text_response)
    ):
        await conn.commit()

async def main():
    conn = await aiosqlite.connect('../data/thumbnail_model.db')

    # Initialize S3 and Rekognition clients
    s3 = boto3.client('s3')
    rekognition = boto3.client('rekognition')
    bucket_name = 'thumbnail-lake'
    
    # get video_ids in statistics table
    existing_ids = %sql SELECT video_id FROM rekognition_responses
    existing_ids = [row[0] for row in existing_ids]
    
    # get video_ids in rekognition_responses table
    existing_keys = %sql SELECT video_id FROM statistics
    existing_keys = [row[0] for row in existing_keys]
    
    # Determine which files need Rekognition analysis
    unanalyzed_ids = [key for key in existing_keys if key not in existing_ids]

    # get s3 keys
    paginator = s3.get_paginator('list_objects_v2')
    page_iterator = paginator.paginate(Bucket=bucket_name)
    s3_keys = [obj['Key'] for page in page_iterator for obj in page.get('Contents', []) if obj['Key']]
    
    # Determine which files need Rekognition analysis
    file_keys = [key for key in s3_keys if key.split('.')[0] in unanalyzed_ids]

    keys_to_analyze = []
    
    for key in file_keys:
        if key not in existing_ids:
            keys_to_analyze.append(key)
            
    for key in tqdm(keys_to_analyze, desc="Analyzing images"):
        image_response = await asyncio.to_thread(get_image_response_sync, bucket_name, key, rekognition)
        label_response = json.dumps(image_response.get('Label Response', {}))
        text_response = json.dumps(image_response.get('Text Response', {}))
        await insert_into_db(conn, key, label_response, text_response)

    # Close the database connection
    await conn.close()

# Run the asyncio event loop
await main()


Analyzing images:   0%|          | 0/493 [00:00<?, ?it/s]

In [6]:
rekognition_df = %sql SELECT * from rekognition_responses;
rekognition_df = rekognition_df.DataFrame().drop_duplicates()
rekognition_df

,video_id,label_response,text_response
0,--eCdoJdTOg.jpg,"{""Labels"": [{""Name"": ""Hunting"", ""Confidence"": ...","{""TextDetections"": [{""DetectedText"": ""ELDEN RI..."
1,-0pX7sE3Als.jpg,"{""Labels"": [{""Name"": ""Book"", ""Confidence"": 99....","{""TextDetections"": [{""DetectedText"": ""ELDEN RI..."
2,-2RaFIQ1G5Y.jpg,"{""Labels"": [{""Name"": ""Head"", ""Confidence"": 99....","{""TextDetections"": [{""DetectedText"": ""US"", ""Ty..."
3,-CBbJLN21yA.jpg,"{""Labels"": [{""Name"": ""Outdoors"", ""Confidence"":...","{""TextDetections"": [], ""TextModelVersion"": ""3...."
4,-CuStFXCtBk.jpg,"{""Labels"": [{""Name"": ""Face"", ""Confidence"": 99....","{""TextDetections"": [{""DetectedText"": ""ELDEN RI..."
...,...,...,...
11678,zS0rV9AG4mY.jpg,"{""Labels"": [{""Name"": ""Plant"", ""Confidence"": 97...","{""TextDetections"": [{""DetectedText"": ""ELDEN RI..."
11679,zhHh7ccysIM.jpg,"{""Labels"": [{""Name"": ""Person"", ""Confidence"": 8...","{""TextDetections"": [], ""TextModelVersion"": ""3...."
11680,zp8JPLeu5pw.jpg,"{""Labels"": [{""Name"": ""Book"", ""Confidence"": 99....","{""TextDetections"": [{""DetectedText"": ""ELDEN RI..."
11681,zpQUM0giSM8.jpg,"{""Labels"": [{""Name"": ""Person"", ""Confidence"": 7...","{""TextDetections"": [{""DetectedText"": ""NAMELESS..."


In [7]:
# statistics_df = %sql SELECT * from statistics;
# statistics_df = statistics_df.DataFrame().drop_duplicates()

# statistics_df.shape[0] - rekognition_df.shape[0]